In [7]:
 ## Probando OpenStreetMap 

import requests

def convert_to_coords_osm(address):
    url = f"https://nominatim.openstreetmap.org/search?q={address}&format=json"
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()  # Lanza un error si hay problemas HTTP (403, 500, etc.)
        data = response.json()
        if not data:
            print("No se encontraron resultados para la dirección.")
            return None
        return data[0]["lat"], data[0]["lon"]
    except requests.exceptions.RequestException as e:
        print(f"Error en la solicitud: {e}")
        return None
    except (KeyError, IndexError):
        print("Error al procesar los datos de la API.")
        return None

coords = convert_to_coords_osm("The Washington Monument, DC")
print(coords)


('38.88759715', '-77.04349684835228')


In [ ]:
from datetime import datetime

import requests

# API Key de OpenRouteService (reemplázala con tu clave)
API_KEY = "5b3ce3597851110001cf624847c65edb6af245adb9946cc4b1b81e91"

def convert_to_coords_osm(address):
    url = f"https://nominatim.openstreetmap.org/search?q={address}&format=json"
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()  # Lanza un error si hay problemas HTTP (403, 500, etc.)
        data = response.json()
        if not data:
            print(f"No se encontraron resultados para la dirección: {address}")
            return None
        return float(data[0]["lat"]), float(data[0]["lon"])  # Retorna (lat, lon)
    except requests.exceptions.RequestException as e:
        print(f"Error en la solicitud: {e}")
        return None
    except (KeyError, IndexError):
        print("Error al procesar los datos de la API.")
        return None


def build_mapping_dict(start, end, waypoints):
    mapping_dict = {}

    start_coords = convert_to_coords_osm(start)
    end_coords = convert_to_coords_osm(end)
    
    if start_coords and end_coords:
        mapping_dict["start"] = start_coords
        mapping_dict["end"] = end_coords
    else:
        print("Error: No se pudo obtener las coordenadas de inicio o fin.")
        return None  # Detiene la ejecución si no hay coordenadas válidas

    if waypoints:
        for i, waypoint in enumerate(waypoints):
            waypoint_coords = convert_to_coords_osm(waypoint)
            if waypoint_coords:
                mapping_dict[f"waypoint_{i}"] = waypoint_coords  # Agrega la tupla (lat, lon)

    return mapping_dict


def build_directions_and_route(mapping_dict, start_time=None, transit_type="driving"):
    if not mapping_dict:
        print("Error: El diccionario de mapeo es inválido.")
        return None

    if not start_time:
        start_time = datetime.now()

    # Convertir a formato [lon, lat] para OpenRouteService
    start = mapping_dict["start"][::-1]  # (lat, lon) → (lon, lat)
    end = mapping_dict["end"][::-1]  # (lat, lon) → (lon, lat)

    waypoints = [
        mapping_dict[x][::-1]  # Convertir cada waypoint de (lat, lon) a (lon, lat)
        for x in mapping_dict.keys() if "waypoint" in x
    ]

    # Construcción de parámetros para OpenRouteService
    params = {
        "api_key": API_KEY,
        "start": f"{start[0]},{start[1]}",  # OpenRouteService usa lon,lat
        "end": f"{end[0]},{end[1]}",
        "waypoints": "|".join([f"{wp[0]},{wp[1]}" for wp in waypoints]) if waypoints else None,
        "mode": transit_type,
    }
    print(params)

    url = "https://api.openrouteservice.org/v2/directions/driving-car"
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error en la solicitud de direcciones: {e}")
        return None
    


import requests


url = "https://api.openrouteservice.org/v2/directions/driving-car"
params = {
    "api_key": API_KEY,
    "start": "-58.3816,-34.6037",  # Buenos Aires
    "end": "-60.6393,-32.9468",    # Rosario
}

response = requests.get(url, params=params).json()
print(response)
    



In [32]:
start = "Buenos Aires, Argentina"
end = "Rosario, Argentina"
waypoints = ["Córdoba, Argentina", "Santa Fe, Argentina"]

mapping_dict = build_mapping_dict(start, end, waypoints)

if mapping_dict:
    route = build_directions_and_route(mapping_dict)
    print(route)

{'api_key': 'TU_API_KEY', 'start': '-58.3816,-34.6037', 'end': '-60.6393,-32.9468'}
{'error': 'Access to this API has been disallowed'}
